In [1]:
import pandas as pd
import numpy as np
import sklearn
# Data preprocessing and cleaning
df = pd.read_csv('https://raw.githubusercontent.com/anvarnarz/praktikum_datasets/main/housing_data_08-02-2021.csv')
# Convert 'size' column to numeric and remove invalid values
df['size'] = pd.to_numeric(df['size'], errors='coerce')
df.dropna(subset=['size'], inplace=True)
df['size'] = df['size'].round()
# Round values and convert to integer
df['size'] = df['size'].astype('int64')
# Convert 'price' column to numeric and remove invalid values
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df.dropna(subset=['price'], inplace=True)
# Round values and convert to integer
df['price'] = df['price'].astype('int64')


# Split the dataset into training and testing sets (80% train, 20% test)
from sklearn.model_selection import train_test_split
train_set,test_set=train_test_split(df,test_size=0.2,random_state=50)
# Separate features (X_train) and target variable (y)
# Drop unnecessary columns like 'location'
x_train=train_set.drop(['price','location'],axis=1)
y = train_set['price'].copy()

# Dropthe column 'distirct' to keep numeric only in our dataframe
x_num = x_train.drop('district',axis=1)

In [2]:
# Custom transformer to create new feature: size_per_room
# This feature represents the average room size
from sklearn.base import BaseEstimator, TransformerMixin
class CombinedAttributes(BaseEstimator, TransformerMixin):
    def __init__(self, add_size_per_room=True):
        self.add_size_per_room = add_size_per_room
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        size_per_room = X['size'] / X['rooms'] # Calculate size_per_room = size / rooms
        if self.add_size_per_room:
            return np.c_[X, size_per_room] # Use np.c_ with X (converted to numpy implicitly) and the new feature
        else:
            return X

# Combine numerical and categorical pipelines using ColumnTransformer
# Numerical pipeline:
# - Fill missing values using median
# - Add new features using custom transformer
# - Scale features using StandardScaler

# Categorical pipeline:
# - Convert 'district' into numerical format using OneHotEncoder

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

num_attribs = ['size', 'rooms', 'level', 'max_levels']
cat_attribs = ['district']

# Instantiate imputer and scaler outside the pipeline to use set_output
imputer = SimpleImputer(strategy='median')
imputer.set_output(transform="pandas")
scaler = StandardScaler()
scaler.set_output(transform="pandas")

num_pipeline = Pipeline([
    ('imputer', imputer),
    ('attribs_adder', CombinedAttributes()),
    ('scaler', scaler)
])

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

In [4]:
# Apply full pipeline to training data
x_prepared = full_pipeline.fit_transform(x_train)

In [5]:
x_prepared

array([[-0.01682566,  1.29557724,  0.58249377, ...,  0.        ,
         0.        ,  0.        ],
       [-0.00788622,  2.23171604, -0.31130191, ...,  0.        ,
         0.        ,  0.        ],
       [-0.03470453,  0.35943843,  1.92318728, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.03470453,  0.35943843, -0.75819975, ...,  0.        ,
         0.        ,  0.        ],
       [-0.03828031, -0.57670038, -1.20509759, ...,  0.        ,
         0.        ,  0.        ],
       [-0.03768434,  0.35943843, -0.31130191, ...,  0.        ,
         0.        ,  0.        ]])

Linear Regression

In [6]:
from sklearn.linear_model import LinearRegression
LR_model = LinearRegression()

In [7]:
LR_model.fit(x_prepared, y)

LinearRegression()

This is a sample data we will check our model with test data that we keep  20%.

In [23]:
y_predicted2 = LR_model.predict(x_test_prepared)

The rusult i take from LR_model did not satisfy me so lets try another models


In [24]:
from sklearn.metrics import mean_squared_error
y_predicted2 = LR_model.predict(x_test_prepared)
lin_mse = mean_squared_error(y_test, y_predicted2)
print('RMSE=',np.sqrt(lin_mse))

RMSE= 181106.84497755708


In [8]:
test_set

,location,district,rooms,size,level,max_levels,price
144,"город Ташкент, Яккасарайский район, Кушбеги",Яккасарайский,1,38,2,5,25600
1464,"город Ташкент, Чиланзарский район, Чиланзар 5 ...",Чиланзарский,3,100,6,9,78000
580,"город Ташкент, Учтепинский район, Чиланзар ква...",Учтепинский,4,80,5,5,62000
762,"город Ташкент, Мирзо-Улугбекский район, Турон",Мирзо-Улугбекский,2,60,4,4,31992
1626,"город Ташкент, Мирзо-Улугбекский район, Ялангач",Мирзо-Улугбекский,3,65,4,4,34998
...,...,...,...,...,...,...,...
6967,"город Ташкент, Учтепинский район, Чиланзар 22-...",Учтепинский,3,65,4,4,42800
5163,"город Ташкент, Чиланзарский район, Чиланзар",Чиланзарский,1,36,3,5,25500
6868,"город Ташкент, Мирабадский район, Янгизамон",Мирабадский,3,71,2,9,47000
3094,"город Ташкент, Мирабадский район, Чехова",Мирабадский,3,105,7,8,95000


In [9]:
x_test = test_set.drop(['price', 'location'], axis=1)
x_test

,district,rooms,size,level,max_levels
144,Яккасарайский,1,38,2,5
1464,Чиланзарский,3,100,6,9
580,Учтепинский,4,80,5,5
762,Мирзо-Улугбекский,2,60,4,4
1626,Мирзо-Улугбекский,3,65,4,4
...,...,...,...,...,...
6967,Учтепинский,3,65,4,4
5163,Чиланзарский,1,36,3,5
6868,Мирабадский,3,71,2,9
3094,Мирабадский,3,105,7,8


In [10]:
y_test = test_set['price'].copy()
y_test

,price
144,25600
1464,78000
580,62000
762,31992
1626,34998
...,...
6967,42800
5163,25500
6868,47000
3094,95000


In [11]:
x_test_prepared = full_pipeline.transform(x_test)

Here i am trying #RandomForestRegressor to see real result

In [12]:
from sklearn.ensemble import RandomForestRegressor
RF_model= RandomForestRegressor()
RF_model.fit(x_prepared, y)

RandomForestRegressor()

In [13]:
y_predicted = RF_model.predict(x_test_prepared)

In [30]:
from sklearn.metrics import median_absolute_error
lin_mae = median_absolute_error(y_test, y_predicted)
print('MAE=',lin_mae)

MAE= 5192.328750000001


In [17]:
import pickle
with open('LR_model.pkl','wb') as file:
    pickle.dump(LR_model,file)

In [18]:
import joblib
joblib.dump(LR_model,'LR_model.pkl')

['LR_model.pkl']